# 03 - Model Training & Evaluation

This notebook trains and evaluates two ML models for ACL injury risk prediction:
- **Random Forest** (primary model)
- **Logistic Regression** (baseline)

Both use balanced class weights and 5-fold stratified cross-validation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from config import PROCESSED_DATA_DIR
from src.features.feature_pipeline import FEATURE_NAMES
from src.models.train import train_pipeline
from src.models.evaluate import (
    evaluate_model, plot_roc_curve, plot_confusion_matrix, plot_feature_importance,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## Load Feature Matrix

In [ ]:
data = np.load(PROCESSED_DATA_DIR / 'feature_matrix.npz', allow_pickle=True)
X = data['X']
y = data['y']
print(f"Feature matrix: {X.shape}")
print(f"Class distribution: healthy={sum(y==0)}, injured={sum(y==1)}")

## Train Models

In [ ]:
results = train_pipeline(X, y)

print("\n=== Cross-Validation Results ===")
for name in ['random_forest', 'logistic_regression']:
    print(f"\n{name}:")
    print(f"  Best CV ROC AUC: {results[name]['cv_score']:.4f}")
    print(f"  Best params: {results[name]['best_params']}")

## Evaluate on Test Set

In [ ]:
X_test = results['X_test']
y_test = results['y_test']

for name in ['random_forest', 'logistic_regression']:
    model = results[name]['model']
    metrics = evaluate_model(model, X_test, y_test, name)
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  ROC AUC:   {metrics['roc_auc']:.4f}")

## ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, ['random_forest', 'logistic_regression']):
    model = results[name]['model']
    y_proba = model.predict_proba(X_test)[:, 1]
    plot_roc_curve(y_test, y_proba, name.replace('_', ' ').title())

plt.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name in zip(axes, ['random_forest', 'logistic_regression']):
    model = results[name]['model']
    y_pred = model.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, name.replace('_', ' ').title())

plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Random Forest feature importance
rf_model = results['random_forest']['model']
plot_feature_importance(rf_model, FEATURE_NAMES, top_n=15)
plt.show()

In [ ]:
# Logistic Regression coefficients
lr_model = results['logistic_regression']['model']
plot_feature_importance(lr_model, FEATURE_NAMES, top_n=15)
plt.show()

## Error Analysis

In [ ]:
# Examine misclassified samples
rf_model = results['random_forest']['model']
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

misclassified = np.where(y_pred != y_test)[0]
print(f"Misclassified: {len(misclassified)} / {len(y_test)} ({len(misclassified)/len(y_test):.1%})")

for idx in misclassified:
    print(f"  Sample {idx}: true={y_test[idx]}, predicted={y_pred[idx]}, probability={y_proba[idx]:.3f}")